In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set working directory
import os
os.chdir('/content/drive/MyDrive/brain-tumor-project/notebooks')

# Verify
!ls ../data/classification/train/

# 12 - Model Comparison & Analysis

Comprehensive comparison of all 10 classification models.
Generates paper-ready tables, charts, and statistical analysis.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

BASE_DIR = os.path.dirname(os.getcwd())
sys.path.append(BASE_DIR)
RESULTS_DIR = os.path.join(BASE_DIR, "results")

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
print(f"Results directory: {RESULTS_DIR}")

## 1. Load All Model Results

In [ ]:
# Load results from all models
results = []
model_keys = [
    "custom_cnn", "vgg16", "vgg19", "resnet50", "resnet101",
    "inceptionv3", "densenet121", "mobilenetv2", "efficientnetb0", "xception"
]

for key in model_keys:
    filepath = os.path.join(RESULTS_DIR, f"{key}_results.json")
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            results.append(json.load(f))
    else:
        print(f"Warning: {filepath} not found")

df = pd.DataFrame(results)
print(f"Loaded results for {len(df)} models")
print(df.to_string(index=False))

## 2. Performance Comparison Table (Paper-Ready)

In [ ]:
# Create formatted comparison table
display_cols = {
    "model": "Model",
    "input_size": "Input Size",
    "total_params": "Parameters",
    "test_accuracy": "Accuracy",
    "test_precision": "Precision",
    "test_recall": "Recall",
    "test_f1": "F1-Score",
    "test_auc": "AUC",
    "training_time_sec": "Train Time (s)"
}

df_display = df[list(display_cols.keys())].rename(columns=display_cols)
df_display = df_display.sort_values("Accuracy", ascending=False)
df_display["Parameters"] = df_display["Parameters"].apply(lambda x: f"{x/1e6:.1f}M")

print("\n" + "="*100)
print("MODEL COMPARISON TABLE (Paper-Ready)")
print("="*100)
print(df_display.to_string(index=False))

# Highlight best model
best_idx = df["test_accuracy"].idxmax()
print(f"\n🏆 Best Model: {df.loc[best_idx, 'model']} (Accuracy: {df.loc[best_idx, 'test_accuracy']:.4f})")

## 3. Performance Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics = ["test_accuracy", "test_precision", "test_recall", "test_f1"]
titles = ["Test Accuracy", "Test Precision", "Test Recall", "Weighted F1-Score"]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(df)))

for ax, metric, title in zip(axes.flat, metrics, titles):
    sorted_df = df.sort_values(metric, ascending=True)
    bars = ax.barh(sorted_df["model"], sorted_df[metric], color=colors)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.grid(axis="x", alpha=0.3)
    
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f"{val:.4f}", 
                va="center", fontsize=9, fontweight="bold")

plt.suptitle("Model Performance Comparison", fontsize=18, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "model_comparison_bars.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Radar Plot

In [ ]:
from matplotlib.patches import FancyBboxPatch

metrics_radar = ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_auc"]
labels = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC"]
num_vars = len(labels)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

colors_radar = plt.cm.tab10(np.linspace(0, 1, len(df)))

for i, (_, row) in enumerate(df.iterrows()):
    values = [row[m] for m in metrics_radar]
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=row["model"], color=colors_radar[i])
    ax.fill(angles, values, alpha=0.05, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title("Model Performance Radar Chart", fontsize=16, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "model_comparison_radar.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Training Time vs Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    df["training_time_sec"] / 60,
    df["test_accuracy"],
    s=df["total_params"] / 1e5,
    c=range(len(df)),
    cmap="viridis",
    alpha=0.7,
    edgecolors="white",
    linewidth=2
)

for _, row in df.iterrows():
    ax.annotate(
        row["model"], 
        (row["training_time_sec"]/60, row["test_accuracy"]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=9
    )

ax.set_xlabel("Training Time (minutes)", fontsize=12)
ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Training Time vs Accuracy (bubble size = parameters)", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "model_comparison_time_vs_acc.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Statistical Summary

In [ ]:
print("="*60)
print("STATISTICAL SUMMARY")
print("="*60)
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_auc"]:
    values = df[metric]
    print(f"\n{metric}:")
    print(f"  Mean: {values.mean():.4f} ± {values.std():.4f}")
    print(f"  Min:  {values.min():.4f} ({df.loc[values.idxmin(), 'model']})")
    print(f"  Max:  {values.max():.4f} ({df.loc[values.idxmax(), 'model']})")

print("\n" + "="*60)
print("PAPER CONCLUSION")
print("="*60)
best = df.loc[df["test_accuracy"].idxmax()]
print(f"\nThe {best['model']} model achieved the highest test accuracy of {best['test_accuracy']:.4f}")
print(f"with a weighted F1-score of {best['test_f1']:.4f} and AUC of {best['test_auc']:.4f}.")
print(f"Total trainable parameters: {best['trainable_params']:,}")
print(f"Training time: {best['training_time_sec']/60:.1f} minutes")

# Export LaTeX table for paper
latex_table = df_display.to_latex(index=False, float_format="%.4f")
with open(os.path.join(RESULTS_DIR, "comparison_table.tex"), "w") as f:
    f.write(latex_table)
print("\nLaTeX table saved to results/comparison_table.tex")